# rust_dataflow

> Fill in a module description here

In [ ]:
# | default_exp rust_dataflow

In [ ]:
# | export
import os
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import jinja2
import networkx as nx
import numpy as np

from spannerflow.config import Config
from spannerflow.engine import Engine, serialize_row
from spannerflow.graph_utils import (
    create_iter_graph,
    find_ingress_nodes,
    find_output,
    find_sources,
    get_common_cols,
    get_minus_cols,
    get_node_schema,
    reduced_graph,
    traverse_cycle,
)
from spannerflow.span import Span

# Code Generation

In [ ]:
# | export
DATAFLOW_TO_RUST_TYPES = {
    "DATA_TYPE_STRING": "String",
    "DATA_TYPE_INT": "i32",
    "DATA_TYPE_FLOAT": "OrderedFloat<f32>",
    "DATA_TYPE_BOOL": "bool",
    "DATA_TYPE_INT64": "i64",
    "DATA_TYPE_SPAN": "Span",
}

PYTHON_TO_DATAFLOW_TYPES = {
    str: "DATA_TYPE_STRING",
    int: "DATA_TYPE_INT",
    float: "DATA_TYPE_FLOAT",
    bool: "DATA_TYPE_BOOL",
    object: "DATA_TYPE_STRING",
    Span: "DATA_TYPE_SPAN",
    np.int64: "DATA_TYPE_INT64",
}

STD_IE_FUNCTIONS: dict[str, dict[tuple, str]] = {
    "rgx": {
        ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_str_span",
        ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_span_span",
    },
    "as_str": {("DATA_TYPE_SPAN",): "span_as_str"},
    "deconstruct_span": {("DATA_TYPE_SPAN",): "deconstruct_span"},
    "rgx_is_match": {
        ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_is_match_str",
        ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_is_match_span",
    },
    "span_contained": {("DATA_TYPE_SPAN", "DATA_TYPE_SPAN"): "span_contained"},
    "rgx_split": {
        ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_split_str",
        ("DATA_TYPE_STRING", "DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_split_str",
        ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_split_span",
        ("DATA_TYPE_STRING", "DATA_TYPE_SPAN", "DATA_TYPE_STRING"): "rgx_split_span",
    },
}

In [ ]:
# | export


def get_sources_data(
    graph: nx.DiGraph,
    code_metadata: dict,  # type: ignore
) -> dict[str | int, dict[str, str | int | list[str]]]:
    data = dict()
    for source in find_sources(graph):
        gr_node = graph.nodes[source]
        op = gr_node["op"]
        data[source] = {
            "name": source,
            "op": op,
        }
        nodes_schema_types_dict = code_metadata.get("nodes_schema_types_dict", {})
        if source not in nodes_schema_types_dict:
            data[source]["schema"] = [
                DATAFLOW_TO_RUST_TYPES[x]
                for x in code_metadata["input_schema"][str(source)]
            ]
        else:
            data[source]["schema"] = [
                DATAFLOW_TO_RUST_TYPES[t] for t in nodes_schema_types_dict[source]
            ]
            if op == "get_const":
                data[source]["consts"] = serialize_row(
                    nodes_schema_types_dict[source],
                    [gr_node["const_dict"][col] for col in gr_node["schema"]],
                )
    return data


def get_col_schema(cols: list[str]) -> str:
    """Return the schema of the columns in the form of a string"""
    if not cols:
        return "0"
    if len(cols) > 1:
        return f"({', '.join(cols)})"
    else:
        return cols[0]


def get_node_str(
    node: str | int, anchor: str | int | None = None, in_iterate: bool = False
) -> str:
    """Return the string representation for the node variable name"""
    if in_iterate and node == anchor:
        return str(anchor)
    return f"node_{node}"


def get_repeatable_cols_in_schema(schema: list[str]) -> dict[str, list[int]]:
    """Return a dictionary of the repeated columns in the schema"""
    repeatable_cols: dict[str, list[int]] = {}
    for i, col in enumerate(schema):
        if col in repeatable_cols:
            repeatable_cols[col].append(i)
        else:
            repeatable_cols[col] = [i]
    repeatable_cols = {
        col: idxs for col, idxs in repeatable_cols.items() if len(idxs) > 1
    }
    return repeatable_cols


def update_repeatable_cols_in_schema(schema: list[str]) -> list[str]:
    """Update the schema to include the index of the repeated columns"""
    repeatable_cols = get_repeatable_cols_in_schema(schema)

    if not repeatable_cols:
        return schema

    result = schema.copy()
    for col, indecies in repeatable_cols.items():
        for i, index in enumerate(indecies):
            result[index] = f"{col}_{i}"
    return result

In [ ]:
# | export


def get_join_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Returns a string of rust code for joining two collections on the common columns.
    If no columns are common, the join is a cross product.
    """
    join1, join2 = list(graph.pred[node])
    out_node_str = get_node_str(node)
    join1_str = get_node_str(join1)
    join2_str = get_node_str(join2)
    if in_iterate:
        if node == anchor:
            out_node_str = str(anchor)
        if join1 == anchor:
            join1_str = join1
        if join2 == anchor:
            join2_str = join2
    common_cols = get_common_cols(graph, join1, join2)
    common_schema = get_col_schema(common_cols)

    join1_uncommon_schema = get_col_schema(get_minus_cols(graph, join1, common_cols))
    join1_schema = (
        get_node_schema(graph, join1) if graph.nodes[join1]["schema"] else "_"
    )
    out_join1_uncommon_schema = (
        join1_uncommon_schema if join1_uncommon_schema != "0" else "_"
    )

    join2_uncommon_schema = get_col_schema(get_minus_cols(graph, join2, common_cols))
    join2_schema = (
        get_node_schema(graph, join2) if graph.nodes[join2]["schema"] else "_"
    )
    out_join2_uncommon_schema = (
        join2_uncommon_schema if join2_uncommon_schema != "0" else "_"
    )

    node_out_schema = get_node_schema(graph, node)
    return f"""let {out_node_str} = {join1_str}.map(|{join1_schema}| ({common_schema}, {join1_uncommon_schema}))
                        .join(&{join2_str}.map(|{join2_schema}| ({common_schema}, {join2_uncommon_schema})))
                        .map(|({common_schema if common_schema != "0" else "_"}, ({out_join1_uncommon_schema}, {out_join2_uncommon_schema}))| ({node_out_schema}));"""

In [ ]:
# test join code
join_sub_graph = nx.DiGraph()
join_sub_graph.add_node(1, op="project", schema=["X", "Y"], rule_id={0})
join_sub_graph.add_node(3, op="project", schema=["Y", "Z"], rule_id={0})
join_sub_graph.add_node(4, op="join", schema=["X", "Y", "Z"], rule_id={0})
join_sub_graph.add_edges_from([(1, 4), (3, 4)])

assert (
    get_join_code(join_sub_graph, 4, {}).strip().replace("  ", "").replace("\n", "")
    == "let node_4 = node_1.map(|(X, Y)| (Y, X)).join(&node_3.map(|(Y, Z)| (Y, Z))).map(|(Y, (X, Z))| ((X, Y, Z)));"
)

# test only common columns
join_sub_graph.nodes[3]["schema"] = ["X", "Y"]
assert (
    get_join_code(join_sub_graph, 4, {}).strip().replace("  ", "").replace("\n", "")
    == "let node_4 = node_1.map(|(X, Y)| ((Y, X), 0)).join(&node_3.map(|(X, Y)| ((Y, X), 0))).map(|((Y, X), (_, _))| ((X, Y, Z)));"
)

# test no common columns
join_sub_graph.nodes[3]["schema"] = ["W", "Z"]
join_sub_graph.nodes[4]["schema"] = ["X", "Y", "W", "Z"]
assert (
    get_join_code(join_sub_graph, 4, {}).strip().replace("  ", "").replace("\n", "")
    == "let node_4 = node_1.map(|(X, Y)| (0, (Y, X))).join(&node_3.map(|(W, Z)| (0, (Z, W)))).map(|(_, ((Y, X), (Z, W)))| ((X, Y, W, Z)));"
)

# test empty schema join (true or false)
join_sub_graph.nodes[3]["schema"] = []
join_sub_graph.nodes[4]["schema"] = ["X", "Y"]
assert (
    get_join_code(join_sub_graph, 4, {}).strip().replace("  ", "").replace("\n", "")
    == "let node_4 = node_1.map(|(X, Y)| (0, (Y, X))).join(&node_3.map(|_| (0, 0))).map(|(_, ((Y, X), _))| ((X, Y)));"
)

In [ ]:
# | export
def get_union_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Returns a string of rust code for unioning two collections."""
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)
    preds = [
        pred
        for pred in graph.pred[node]
        if "reduced" not in graph.get_edge_data(pred, node)
    ]
    var_decl = f"let{' mut' if not in_iterate and node_str == 'node_' + str(node) else ''} {node_str}"
    if not preds:
        return f"{var_decl} = scope.new_collection_from(vec![]).1;"
    prev_node1_str = get_node_str(preds[0], anchor=anchor, in_iterate=in_iterate)
    if len(preds) == 1:
        return f"{var_decl} = {prev_node1_str}.clone();"

    uninion_code = prev_node1_str
    for i in range(1, len(preds)):
        prev_node_i_str = get_node_str(preds[i], anchor=anchor, in_iterate=in_iterate)
        uninion_code = f"{uninion_code}.concat(&{prev_node_i_str})"

    return f"{var_decl} = {uninion_code}.distinct();"

In [ ]:
# test union code
union_graph = nx.DiGraph()
union_graph.add_nodes_from(
    [
        (2, {"op": "project", "schema": ["X", "Y"], "rule_id": {0}}),
        (5, {"op": "project", "schema": ["X", "Y"], "rule_id": {1}}),
        ("A", {"op": "union", "schema": ["col_0", "col_1"], "rule_id": {0, 1}}),
    ]
)
union_graph.add_edges_from([(2, "A"), (5, "A")])
print(get_union_code(union_graph, "A", {}))
assert (
    get_union_code(union_graph, "A", {})
    == "let mut node_A = node_2.concat(&node_5).distinct();"
)
union_graph.nodes[5]["schema"] = ["Y", "Z"]
assert (
    get_union_code(union_graph, "A", {})
    == "let mut node_A = node_2.concat(&node_5).distinct();"
)

single_union_graph = nx.DiGraph()
single_union_graph.add_nodes_from(
    [
        (2, {"op": "project", "schema": ["X", "Y"], "rule_id": {0}}),
        ("A", {"op": "union", "schema": ["X", "Y"], "rule_id": {0}}),
    ]
)
single_union_graph.add_edges_from([(2, "A")])
assert get_union_code(single_union_graph, "A", {}) == "let mut node_A = node_2.clone();"

let mut node_A = node_2.concat(&node_5).distinct();


In [ ]:
# | export


def get_project_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Generation of rust code for the project operation using differetial dataflow operator - map"""
    prev_nodes = list(graph.pred[node])
    prev_node_str = get_node_str(prev_nodes[0], anchor=anchor, in_iterate=in_iterate)
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)
    prev_schema = get_node_schema(graph, prev_nodes[0])

    col_names = graph.nodes[node]["schema"].copy()
    repeatable_cols = get_repeatable_cols_in_schema(graph.nodes[node]["schema"])

    for col_idx in repeatable_cols.values():
        if (
            code_metadata["nodes_schema_types_dict"][node][col_idx[0]]
            != "DATA_TYPE_STRING"
        ):
            continue
        for idx in col_idx:
            col_names[idx] = f"{col_names[idx]}.clone()"

    return f"let {node_str} = {prev_node_str}.map(|{prev_schema}| {get_col_schema(col_names)});"

In [ ]:
# Test project code
graph = nx.DiGraph()
graph.add_nodes_from(
    [
        (1, {"op": "rename", "schema": ["X", "_F1"]}),
        (2, {"op": "project", "schema": ["X"], "rule_id": {0}}),
    ]
)
graph.add_edges_from([(1, 2)])
assert (
    get_project_code(graph, 2, {"nodes_schema_types_dict": {2: ["DATA_TYPE_INT"]}})
    == "let node_2 = node_1.map(|(X, _F1)| X);"
)

In [ ]:
# | export


def get_from_input_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)
    return f"let {node_str} = input_{node}.to_collection(scope);"

In [ ]:
# | export


def get_rename_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Generation of rust code for the rename operation using differetial dataflow operator"""
    schema = get_col_schema(
        update_repeatable_cols_in_schema(graph.nodes[node]["schema"])
    )

    prev_nodes = list(graph.pred[node])

    prev_node_str = get_node_str(prev_nodes[0], anchor=anchor, in_iterate=in_iterate)
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)

    return f"let {node_str} = {prev_node_str}.map(|{schema}| {schema});"

In [ ]:
# | export


def get_select_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    gr_node = graph.nodes[node]
    prev_nodes = list(graph.pred[node])

    prev_node_str = get_node_str(prev_nodes[0], anchor=anchor, in_iterate=in_iterate)
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)

    theta = gr_node["theta"]
    predicates = []
    if hasattr(theta, "pos_val_tuples"):  #  equalConstTheta
        predicates = [
            (
                f'*col_{pos} == "{val}".to_string()'
                if isinstance(val, str)
                else (
                    f"*col_{pos} == {val}".lower()
                    if isinstance(val, bool)
                    else f"*col_{pos} == {val}"
                )
            )
            for pos, val in theta.pos_val_tuples
        ]
    elif hasattr(theta, "col_pos_tuples"):  #  equalColTheta
        predicates = [
            f"col_{pos1} == col_{pos2}" for pos1, pos2 in theta.col_pos_tuples
        ]
    else:
        raise ValueError(f"Unsupported theta join: {theta}. {dir(theta)}")
    return f"let {node_str} = {prev_node_str}.filter(|{get_col_schema([f"col_{i}" for i in range(len(gr_node['schema']))])}| {' && '.join(predicates)});"

In [ ]:
# | export


def get_groupby_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    gr_node = graph.nodes[node]
    prev_nodes = list(graph.pred[node])
    prev_node_str = get_node_str(prev_nodes[0], anchor=anchor, in_iterate=in_iterate)
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)
    agg = gr_node["agg_names"]
    schema = update_repeatable_cols_in_schema(gr_node["schema"])
    groupby_cols = [schema[i] for i, agg_func in enumerate(agg) if agg_func is None]
    nodes_schema_types_dict = code_metadata["nodes_schema_types_dict"]
    agg_by_cols = {
        schema[i]: {
            "agg_func": agg_func,
            "dataflow_type": nodes_schema_types_dict[node][i],
        }
        for i, agg_func in enumerate(agg)
        if agg_func is not None
    }
    agg_cols = list(agg_by_cols.keys())
    agg_by_index = {}
    for key, agg_func in agg_by_cols.items():
        agg_func["index"] = agg_cols.index(key)
        agg_by_index[agg_func["index"]] = {
            "agg_func": agg_func["agg_func"],
            "col_name": key,
            "dataflow_type": agg_func["dataflow_type"],
        }
    multi_agg = False
    if len(agg_by_cols.values()) > 1:
        multi_agg = True
    declares = []
    agg_code = []
    output = []
    for col_name, agg_func in agg_by_cols.items():
        agg_index = agg_func["index"]
        agg_var = f"{agg_func['agg_func']}_{col_name}"
        output.append(agg_var)
        val = "*val" if not multi_agg else f"val.{agg_index}"
        match agg_func["agg_func"]:
            case "sum":
                declares.append(f"let mut {agg_var}: i32 = 0;")
                agg_code.append(f"{agg_var} += {val} * (*cnt as i32);")
            case "count":
                declares.append(f"let mut {agg_var}: i32 = 0;")
                agg_code.append(f"{agg_var} += *cnt as i32;")
            case "max":
                declares.append(f"let mut {agg_var}: i32 = i32::MIN;")
                agg_code.append(
                    f"{agg_var} = std::cmp::max({agg_var}, {"*" if not multi_agg else ""}{val});"
                )
            case "min":
                declares.append(f"let mut {agg_var}: i32 = i32::MAX;")
                agg_code.append(
                    f"{agg_var} = std::cmp::min({agg_var}, {"*" if not multi_agg else ""}{val});"
                )
            case "avg":
                declares.append(f"let mut {agg_var}: (i32, i32) = (0, 0);")
                agg_code.append(f"{agg_var}.0 += {val} * (*cnt as i32);")
                agg_code.append(f"{agg_var}.1 += *cnt as i32;")
            case _:
                remote_aggregation_template = code_metadata[
                    "template_env"
                ].get_template("remote_aggregate.rs.jinja2")
                code = remote_aggregation_template.render(
                    output_node=node_str,
                    prev_node=prev_node_str,
                    grpc_address=code_metadata["grpc_address"],
                    function_name=agg_func["agg_func"],
                    in_schema=get_col_schema(
                        gr_node["schema"][: len(gr_node["schema"])]
                    ),
                    out_schema_len=len(gr_node["schema"]),
                    in_schema_len=len(gr_node["schema"]),
                    out_schema_types=nodes_schema_types_dict[node],
                    DATAFLOW_TO_RUST_TYPES=DATAFLOW_TO_RUST_TYPES,
                    groupby_schema=get_col_schema(groupby_cols),
                    groupby_len=len(groupby_cols),
                    agg_len=len(agg_cols),
                    agg_schema=get_col_schema(agg_cols),
                    agg_by_index=agg_by_index,
                )
                return code
    agg_template = code_metadata["template_env"].get_template("aggregate.rs.jinja2")
    code = agg_template.render(
        output_node=node_str,
        prev_node=prev_node_str,
        in_schema=get_col_schema(schema),
        mid_schema=get_col_schema(groupby_cols + agg_cols),
        groupby_schema=get_col_schema(groupby_cols),
        groupby_len=len(groupby_cols),
        agg_len=len(agg_cols),
        agg_schema=get_col_schema(agg_cols),
        agg_decleration=declares,
        agg_list=agg_code,
        output_tuple=get_col_schema(output),
        agg_by_index=agg_by_index,
    )
    return code

In [ ]:
# | export


def get_ie_map_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Returns a string of rust code for ie_map node in the graph."""
    prev_nodes = list(graph.pred[node])
    node_str = get_node_str(node, anchor=anchor, in_iterate=in_iterate)
    prev_node_str = get_node_str(prev_nodes[0], anchor=anchor, in_iterate=in_iterate)
    nodes_schema_types_dict = code_metadata["nodes_schema_types_dict"]
    gr_node: dict = graph.nodes[node]
    in_arity: int = gr_node["in_arity"]
    out_arity: int = gr_node["out_arity"]
    in_schema = gr_node["schema"][:in_arity]
    in_type_schema: list = nodes_schema_types_dict[node][:in_arity]
    out_schema = gr_node["schema"][in_arity:]

    match gr_node["name"]:
        case "rgx":
            func_name = STD_IE_FUNCTIONS["rgx"][tuple(in_type_schema)]
            code = f"let {node_str} = {prev_node_str}.flat_map(|{get_col_schema(in_schema)}| {{ \n \
                {func_name}({', '.join([f'&{col}' for col in in_schema])}).map(move |vec| ({', '.join([f'{col}.clone()' for col in in_schema]+[f'vec[{i}].clone()' for i in range(gr_node["out_arity"])])})) \n \
            }});"
        case "rgx_split":
            func_name = STD_IE_FUNCTIONS["rgx_split"][tuple(in_type_schema)]
            func_args = ", ".join([f"&{col}" for col in in_schema])
            if in_arity == 2:
                func_args += ', "Start Tag"'
            code = f"let {node_str} = {prev_node_str}.flat_map(|{get_col_schema(in_schema)}| {{ \n \
                {func_name}({func_args}).map(move |{get_col_schema(out_schema)}| ({', '.join([f'{col}.clone()' for col in in_schema]+out_schema)})) \n \
            }});"

        case (
            "as_str"
            | "rgx_split"
            | "rgx_is_match"
            | "deconstruct_span"
            | "span_contained" as original_func_name
        ):
            func_name = STD_IE_FUNCTIONS[original_func_name][tuple(in_type_schema)]
            code = f"let {node_str} = {prev_node_str}.flat_map(|{get_col_schema(in_schema)}| {{ \n \
                {func_name}({', '.join([f'&{col}' for col in in_schema])}).map(move |{get_col_schema(out_schema)}| ({', '.join([f'{col}.clone()' for col in in_schema]+out_schema)})) \n \
            }});"

        case "not":
            code = f"let {node_str} = {prev_node_str}.map(|{get_node_schema(graph, prev_nodes[0])}| ({get_node_schema(graph, prev_nodes[0])}, !{get_node_schema(graph, prev_nodes[0])}));"
        case _:
            ie_map_template = code_metadata["template_env"].get_template(
                "ie_map.rs.jinja2"
            )
            code = ie_map_template.render(
                output_node=node_str,
                prev_node=prev_node_str,
                grpc_address=code_metadata["grpc_address"],
                function_name=graph.nodes[node]["name"],
                in_schema=get_col_schema(
                    gr_node["schema"][: len(gr_node["in_schema"])]
                ),
                out_schema_len=len(gr_node["out_schema"]),
                in_schema_len=len(gr_node["in_schema"]),
                in_arity=in_arity,
                out_arity=out_arity,
                out_schema_types=nodes_schema_types_dict[node],
                DATAFLOW_TO_RUST_TYPES=DATAFLOW_TO_RUST_TYPES,
            )
    return code

In [ ]:
# | export


def validate_node(graph: nx.DiGraph, node: str | int) -> None:
    """Validates the node in the graph."""
    gr_node: dict = graph.nodes[node]
    preds = [graph.nodes[key] for key in graph.pred[node].keys()]
    match gr_node["op"]:
        case "union":
            preds = [
                pred
                for pred in graph.pred[node]
                if "reduced" not in graph.get_edge_data(pred, node)
            ]
            if len(preds) == 0 and not gr_node.get("anchor", False):
                raise ValueError(
                    "Union node has invalid number of predecessors: ",
                    (len(preds), node),
                )
        case "rename" | "select" | "project" | "groupby":
            if len(preds) != 1:
                raise ValueError(
                    f"{gr_node['op']} node has invalid number of predecessors: ",
                    (len(preds), node),
                )
        case "product" | "join":
            if len(preds) != 2:
                raise ValueError(
                    f"Product {gr_node['op']} has invalid number of predecessors: ",
                    (len(preds), node),
                )
        case "ie_map":
            if gr_node["name"] == "not":
                if len(preds) != 1:
                    raise ValueError(
                        "Not node has invalid number of predecessors: ",
                        (len(preds), node),
                    )
        case "get_const" | "get_rel":
            return
        case _:
            raise ValueError(f"Unsupported operation: {gr_node['op']}")

In [ ]:
# | export


def prepare_node(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
) -> None:
    """Prepares the node in the graph, calculating and updating the schema types dictionary."""
    nodes_schema_types_dict = code_metadata["nodes_schema_types_dict"]
    gr_node = graph.nodes[node]
    preds = [
        {**graph.nodes[node_name], "node_name": node_name}
        for node_name in graph.pred[node]
    ]
    if node in nodes_schema_types_dict:
        return
    match gr_node["op"]:
        case "get_rel":
            nodes_schema_types_dict[node] = code_metadata["input_schema"][node]
        case "rename" | "select" | "union":
            nodes_schema_types_dict[node] = nodes_schema_types_dict[
                preds[0]["node_name"]
            ]
        case "project":
            pred = preds[0]
            nodes_schema_types_dict[node] = [
                nodes_schema_types_dict[pred["node_name"]][pred["schema"].index(col)]
                for col in gr_node["schema"]
            ]
        case "product" | "join":
            schema_types = []
            for x in gr_node["schema"]:
                try:
                    index = preds[0]["schema"].index(x)
                    schema_types.append(
                        nodes_schema_types_dict[preds[0]["node_name"]][index]
                    )
                except ValueError:
                    index = preds[1]["schema"].index(x)
                    schema_types.append(
                        nodes_schema_types_dict[preds[1]["node_name"]][index]
                    )
            nodes_schema_types_dict[node] = schema_types
        case "groupby":
            schema_types = []
            for index, agg in enumerate(gr_node["agg_names"]):
                if agg is None:
                    schema_types.append(
                        nodes_schema_types_dict[preds[0]["node_name"]][index]
                    )
                elif (
                    agg == "count"
                ):  # what about min / max should it match int if the field is int?
                    schema_types.append("DATA_TYPE_INT")
                elif agg in ("sum", "avg", "min", "max"):
                    schema_types.append("DATA_TYPE_FLOAT")
                else:  # custom aggregation function
                    # TODO: Fix python agg function
                    schema_types.append("DATA_TYPE_STRING")
            nodes_schema_types_dict[node] = schema_types
        case "get_const":
            nodes_schema_types_dict[node] = [
                PYTHON_TO_DATAFLOW_TYPES[type(const)]
                for const in gr_node["const_dict"].values()
            ]
        case "ie_map":
            if callable(gr_node["in_schema"]):
                gr_node["in_schema"] = gr_node["in_schema"](gr_node["in_arity"])
            if callable(gr_node["out_schema"]):
                gr_node["out_schema"] = gr_node["out_schema"](gr_node["out_arity"])
            nodes_schema_types_dict[node] = []
            for i, col_type in enumerate(gr_node["in_schema"] + gr_node["out_schema"]):
                if i < gr_node["in_arity"]:
                    prev_type = nodes_schema_types_dict[preds[0]["node_name"]][i]
                    if isinstance(col_type, tuple):
                        tuple_types = (PYTHON_TO_DATAFLOW_TYPES[t] for t in col_type)
                        if prev_type not in tuple_types:
                            raise ValueError(
                                "Type mismatch: the IE function input type does not match the input schema"
                            )
                    new_col_type = prev_type
                else:
                    new_col_type = PYTHON_TO_DATAFLOW_TYPES[col_type]
                nodes_schema_types_dict[node].append(new_col_type)
        case _:
            raise ValueError(f"Unsupported operation: {gr_node['op']}")

In [ ]:
# | export


def generate_node_code(
    graph: nx.DiGraph,
    node: str | int,
    code_metadata: dict,  # type: ignore
    anchor: str | int | None = None,
    in_iterate: bool = False,
) -> str:
    """Returns a string of rust code for the node in the graph."""
    gr_node = graph.nodes[node]
    validate_node(graph, node)
    prepare_node(graph, node, code_metadata=code_metadata)
    op_to_code_generator = {
        "get_rel": get_from_input_code,
        "rename": get_rename_code,
        "project": get_project_code,
        "join": get_join_code,
        "select": get_select_code,
        "union": get_union_code,
        "groupby": get_groupby_code,
        "get_const": get_from_input_code,
        "product": get_join_code,
        "ie_map": get_ie_map_code,
    }
    if gr_node["op"] not in op_to_code_generator:
        raise ValueError(f"Unsupported operation: {gr_node['op']}")
    return op_to_code_generator[gr_node["op"]](
        graph,
        node,
        code_metadata=code_metadata,
        anchor=anchor,
        in_iterate=in_iterate,
    )

In [ ]:
# | export


def generate_graph_code(
    graph: nx.DiGraph,
    code_metadata: dict,  # type: ignore
) -> list[str]:
    flow_code = list()
    iterate_template = code_metadata["template_env"].get_template("iterate.rs.jinja2")
    nodes_schema_types_dict = code_metadata["nodes_schema_types_dict"]
    reduced, cycles = reduced_graph(graph)
    for node in list(nx.topological_sort(reduced)):
        prepare_node(
            reduced,
            node,
            code_metadata=code_metadata,
        )
        if node in cycles.keys():
            nodes_schema_types_dict[f"iter_{node}"] = nodes_schema_types_dict[node]
            prepare_node(
                cycles[node],
                f"iter_{node}",
                code_metadata=code_metadata,
            )
            iter_graph = create_iter_graph(graph, cycles[node], node)
            anchor_code = generate_node_code(reduced, node, code_metadata=code_metadata)
            cycle_code = {}
            cycle_order = traverse_cycle(cycles[node], f"iter_{node}")
            for cycle_node in cycle_order:
                prepare_node(
                    iter_graph,
                    cycle_node,
                    code_metadata=code_metadata,
                )
                cycle_code[cycle_node] = generate_node_code(
                    iter_graph,
                    cycle_node,
                    code_metadata=code_metadata,
                    anchor=f"iter_{node}",
                    in_iterate=True,
                )
            flow_code.append(
                iterate_template.render(
                    {
                        "ingress_nodes": find_ingress_nodes(
                            graph, list(cycles[node].nodes)
                        ),
                        "anchor": node,
                        "cycle_flow": cycle_order,
                        "flow_code": cycle_code,
                        "anchor_code": anchor_code,
                    }
                )
            )
        else:
            flow_code.append(
                generate_node_code(reduced, node, code_metadata=code_metadata)
            )
    return flow_code

# Class RustDataflow

In [ ]:
# | export


class RustDataflow:
    def __init__(self, engine: Engine, config: Config):
        self._engine = engine
        self._config = config
        self._query_id = 0
        self._cargo_file_name = "Cargo.toml"
        self._template_loader = jinja2.FileSystemLoader(
            searchpath=self._config.TEMPLATES_PATH
        )
        self._template_env = jinja2.Environment(loader=self._template_loader)

    def get_input_schema_types(self, node: int | str) -> list[str]:
        collections = self._engine.get_collections()
        return [x for x in collections[str(node)]]

    def get_input_schema(self, node: int | str) -> list[str]:
        collections = self._engine.get_collections()
        return [DATAFLOW_TO_RUST_TYPES[x] for x in collections[str(node)]]

    def create_rust_file(self, timestamp: str, graph: nx.DiGraph) -> None:
        dest_path = self._config.GENERATED_RUST_PROJECT_PATH / f"{timestamp}.rs"
        template = self._template_env.get_template(self._config.RUST_FILE_TEMPLATE_NAME)
        nodes_schema_types_dict: dict[str | int, list[str]] = dict()

        code_metadata = {
            "nodes_schema_types_dict": nodes_schema_types_dict,
            "template_env": self._template_env,
            "input_schema": self._engine.get_collections(),
            "grpc_address": f"http://{self._config.LISTEN_ADDRESS}",
        }

        flow_code = generate_graph_code(graph, code_metadata)
        output_node = find_output(graph)
        output_text = template.render(
            query_id=self._query_id,
            sources=get_sources_data(graph, code_metadata=code_metadata),
            flow_code=flow_code,
            output_node_str=get_node_str(output_node),
            output_vars_count=len(graph.nodes[output_node]["schema"]),
        )
        self._config.RUST_SERVER_LIB_SRC_FILE_PATH.unlink(missing_ok=True)
        with open(dest_path, "w") as f:
            f.write(output_text)

        with open(self._config.RUST_SERVER_LIB_SRC_FILE_PATH, "w") as f:
            f.write(output_text)

        graph.graph["nodes_schema_types_dict"] = nodes_schema_types_dict

    def build_so(self, graph: nx.DiGraph) -> tuple[Path, str]:
        self._config.GENERATED_RUST_PROJECT_PATH.joinpath("src").mkdir(
            parents=True, exist_ok=True
        )
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._query_id += 1
        self.create_rust_file(timestamp, graph)
        self.build_rust(
            self._config.PACKAGE_ROOT.joinpath(self._cargo_file_name).absolute(),
            self._config.RUST_SO_BUILD_LOG_PATH,
        )
        # Determine file extension based on the platform
        crate_name = "spannerflow"
        if os.name == "posix":  # Linux/macOS
            if os.uname().sysname == "Darwin":
                extension = ".dylib"  # macOS
            else:
                extension = ".so"  # Linux
            lib_filename = f"lib{crate_name}{extension}"
        elif os.name == "nt":  # Windows
            extension = ".dll"
            lib_filename = f"{crate_name}{extension}"
        else:
            raise RuntimeError("Unsupported OS")
        lib_path = self._config.PACKAGE_ROOT.joinpath("target", "release", lib_filename)
        new_lib_path = self._config.GENERATED_RUST_PROJECT_PATH.joinpath(
            f"{timestamp}{extension}"
        )
        shutil.copy(lib_path, new_lib_path)
        return (
            new_lib_path,
            f"query_{self._query_id}",
        )

    def build_rust(self, cargo_toml_path: Path, log_path: Path) -> None:
        command = [
            "cargo",
            "build",
            "--manifest-path",
            "--release",
            str(cargo_toml_path),
            "-p",
            "spannerflow",
        ]

        self._config.LOGS_DIR.mkdir(parents=True, exist_ok=True)
        env = os.environ.copy()
        env["RUSTFLAGS"] = f"-C prefer-dynamic -C link-arg=-Wl,-rpath,{self._config.PACKAGE_ROOT.joinpath('target', 'release', 'deps')}"
        with open(log_path, "a") as log_file:
            subprocess.run(
                command,
                cwd=str(cargo_toml_path.parent),
                check=True,
                env=env,
                stderr=log_file,
                stdout=log_file,
            )

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()